# RKPCN Diagnostics for VSEM Experiment

Single-replicate analysis to understand RKPCN behavior across different $\rho$ values.

**Goal**: Understand why higher $\rho$ values appear to produce worse EP approximations in the VSEM example.

**Key questions**:
1. How quickly does the f-chain mix for different $\rho$?
2. What is the effective sample size (ESS) for u-samples at each $\rho$?
3. Does the log-density trace show the chain stuck in modes?
4. Would more iterations fix the issue for high $\rho$?

In [ ]:
from jax import config
config.update('jax_enable_x64', True)

import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from typing import NamedTuple

from uncprop.core.samplers import (
    mcmc_loop, init_rkpcn_kernel, _f_update_pcn_proposal, sample_distribution
)
from uncprop.utils.diagnostics import compute_ess
from uncprop.models.vsem.surrogate import VSEMPosteriorSurrogate, LogDensClippedGPSurrogate

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['font.size'] = 12

## 1. Setup: Reconstruct a replicate

We reconstruct a single replicate from the full experiment run, then re-run RKPCN
with full trace output (not just thinned samples).

In [ ]:
# --- Configuration ---
# Point this at your experiment output
EXPERIMENT_NAME = 'vsem'  # or 'vsem_local_test' for local testing
SURROGATE_TAG = 'clip_gp'  # 'gp' or 'clip_gp'
N_DESIGN = 4
JITTER = 1e-4  # must match the jitter used for this setup
REP_IDX = 0

repo_root = Path('.').resolve().parents[1]
base_dir = repo_root / 'out' / EXPERIMENT_NAME
setup_name = f'{SURROGATE_TAG}_N{N_DESIGN}'
rep_dir = base_dir / setup_name / f'rep{REP_IDX}'

print(f'Rep directory: {rep_dir}')
print(f'Files: {[f.name for f in rep_dir.iterdir()]}')

In [ ]:
# Load the replicate
from experiment import VSEMReplicate
from uncprop.utils.experiment import Experiment

# Read the base key and reconstruct
base_key = jnp.load(base_dir / 'base_key.npy')
base_key = jr.wrap_key_data(base_key)

# Determine the rep key
from uncprop.utils.experiment import Experiment
num_reps = 100
rep_keys = jr.split(base_key, num_reps)
rep_key = rep_keys[REP_IDX]
key_setup, key_run = jr.split(rep_key)

# Reconstruct replicate (this re-fits the surrogate — takes ~10s)
print('Reconstructing replicate...')
surrogate_tag_map = {'gp': 'gp', 'clip_gp': 'clip_gp'}

rep = VSEMReplicate(
    key=key_setup,
    out_dir=rep_dir,
    n_design=N_DESIGN,
    surrogate_tag=SURROGATE_TAG,
    jitter=JITTER,
    write_to_file=False,
)
print('Done.')

In [ ]:
# Load saved samples for reference
saved_samp = dict(jnp.load(rep_dir / 'samples.npz'))
saved_diag = dict(jnp.load(rep_dir / 'diagnostics.npz'))

print('Saved sample keys:', list(saved_samp.keys()))
print('\nSaved diagnostics:')
for k, v in saved_diag.items():
    print(f'  {k}: {float(v):.4f}')

## 2. Re-run RKPCN with full trace output

We re-run each $\rho$ value and keep the full MCMC trace (positions, log-densities, accept probs)
instead of just the thinned samples.

In [ ]:
def run_rkpcn_full_trace(key, rep, rho, n_total=55_000):
    """
    Run RKPCN and return full (unthinned) trace.
    
    Returns dict with:
        positions: (n_total, d) — u positions
        logdensities: (n_total,) — log-density values
        accept_probs: (n_total,) — MH acceptance probabilities
        is_accepted: (n_total,) — whether each proposal was accepted
    """
    key_ker, key_samp = jr.split(key)
    
    surr = rep.posterior_surrogate
    low, high = surr.support
    upper_bound = lambda u: rep.posterior.prior.log_density(u) + rep.posterior.likelihood.log_density_upper_bound(u)

    if isinstance(surr, LogDensClippedGPSurrogate):
        def log_density(f, u):
            u = jnp.atleast_2d(u)
            upper = upper_bound(u)
            lp = jnp.clip(f, max=upper)
            lp = jnp.where(jnp.all((u >= low) & (u <= high), axis=1), lp, -jnp.inf)
            return lp.squeeze()
    else:
        def log_density(f, u):
            u = jnp.atleast_2d(u)
            lp = f
            lp = jnp.where(jnp.all((u >= low) & (u <= high), axis=1), lp, -jnp.inf)
            return lp.squeeze()

    gp = surr.surrogate
    
    class UpdateInfo(NamedTuple):
        rho: float
    f_update_info = UpdateInfo(rho=rho)

    init_fn, kernel = init_rkpcn_kernel(
        key=key_ker, log_density=log_density, gp=gp,
        f_update_fn=_f_update_pcn_proposal, f_update_info=f_update_info)

    # Use the exact posterior's adapted proposal (matches what experiment does)
    initial_position = rep.posterior.prior.sample(jr.key(0))
    
    # Get adapted proposal from exact posterior MCMC
    key_exact = jr.key(999)
    exact_results = sample_distribution(
        key=key_exact, dist=rep.posterior, initial_position=initial_position,
        n_samples=1000, n_burnin=5000, thin_window=1)
    prop_cov = exact_results['prop_cov']
    
    initial_state = init_fn(key_ker, jnp.squeeze(initial_position), prop_cov)
    
    print(f'  Running RKPCN (rho={rho}, n_total={n_total})...')
    states, infos = mcmc_loop(key=key_samp, kernel=kernel,
                              initial_state=initial_state, num_samples=n_total)
    print(f'  Done. Accept rate: {float(jnp.mean(infos.accept_prob)):.4f}')
    
    return {
        'positions': np.array(states.position),
        'logdensities': np.array(states.logdensity),
        'accept_probs': np.array(infos.accept_prob),
        'is_accepted': np.array(infos.is_accepted),
    }

In [ ]:
# Run RKPCN for each rho value
rho_vals = [0.0, 0.9, 0.95, 0.99]
n_total = 55_000  # 50k burnin + 5k post-burnin (matches experiment settings)
n_burnin = 50_000

traces = {}
for rho in rho_vals:
    jax.clear_caches()
    key_rho = jr.key(int(rho * 1000) + 42)
    traces[rho] = run_rkpcn_full_trace(key_rho, rep, rho, n_total=n_total)

print('\nAll runs complete.')

## 3. Log-density trace plots

Shows whether the chain is stuck in modes or exploring smoothly.

In [ ]:
fig, axes = plt.subplots(len(rho_vals), 1, figsize=(14, 3*len(rho_vals)), sharex=True)

for ax, rho in zip(axes, rho_vals):
    ld = traces[rho]['logdensities']
    ax.plot(ld, linewidth=0.3, alpha=0.7)
    ax.axvline(n_burnin, color='red', linestyle='--', linewidth=1, label='burnin end')
    ax.set_ylabel(f'ρ={rho}')
    ax.set_title(f'Log-density trace (ρ={rho})', fontsize=11)
    if rho == rho_vals[0]:
        ax.legend()

axes[-1].set_xlabel('Iteration')
fig.tight_layout()
plt.show()

In [ ]:
# Zoomed: post-burnin only
fig, axes = plt.subplots(len(rho_vals), 1, figsize=(14, 3*len(rho_vals)), sharex=True)

for ax, rho in zip(axes, rho_vals):
    ld = traces[rho]['logdensities'][n_burnin:]
    ax.plot(ld, linewidth=0.5, alpha=0.7)
    ax.set_ylabel(f'ρ={rho}')
    ax.set_title(f'Log-density trace — post-burnin (ρ={rho})', fontsize=11)

axes[-1].set_xlabel('Iteration (post-burnin)')
fig.tight_layout()
plt.show()

## 4. u-parameter trace plots

In [ ]:
par_names = ['av', 'veg_init']

for d_idx, par_name in enumerate(par_names):
    fig, axes = plt.subplots(len(rho_vals), 1, figsize=(14, 3*len(rho_vals)), sharex=True)
    fig.suptitle(f'Trace: {par_name}', fontsize=14, y=1.01)

    for ax, rho in zip(axes, rho_vals):
        pos = traces[rho]['positions'][n_burnin:, d_idx]
        ax.plot(pos, linewidth=0.3, alpha=0.7)
        ax.set_ylabel(f'ρ={rho}')

    axes[-1].set_xlabel('Iteration (post-burnin)')
    fig.tight_layout()
    plt.show()

## 5. Acceptance rate over time

Rolling acceptance rate to see if it stabilizes.

In [ ]:
window = 500

fig, axes = plt.subplots(len(rho_vals), 1, figsize=(14, 3*len(rho_vals)), sharex=True)

for ax, rho in zip(axes, rho_vals):
    acc = traces[rho]['accept_probs']
    rolling = np.convolve(acc, np.ones(window)/window, mode='valid')
    ax.plot(rolling, linewidth=0.5)
    ax.axvline(n_burnin, color='red', linestyle='--', linewidth=1)
    ax.set_ylabel(f'ρ={rho}')
    ax.set_title(f'Rolling accept rate (window={window}, ρ={rho})', fontsize=11)

axes[-1].set_xlabel('Iteration')
fig.tight_layout()
plt.show()

## 6. Effective Sample Size (ESS)

In [ ]:
print(f'{"rho":>6s} | {"ESS(u1)":>10s} {"ESS(u2)":>10s} | {"min ESS":>10s} | {"accept":>8s} | {"n_post":>8s}')
print('-' * 70)

for rho in rho_vals:
    pos = traces[rho]['positions'][n_burnin:]
    ess = compute_ess(pos)
    acc = np.mean(traces[rho]['accept_probs'][n_burnin:])
    n_post = pos.shape[0]
    print(f'{rho:6.2f} | {ess[0]:10.1f} {ess[1]:10.1f} | {min(ess):10.1f} | {acc:8.4f} | {n_post:8d}')

## 7. Autocorrelation analysis

Autocorrelation of u-parameters at different lags — shows how quickly the chain decorrelates.

In [ ]:
def autocorrelation(x, max_lag=500):
    """Compute autocorrelation for a 1D sequence."""
    x = x - np.mean(x)
    n = len(x)
    var = np.var(x)
    if var < 1e-15:
        return np.zeros(max_lag)
    acf = np.correlate(x, x, mode='full')
    acf = acf[n-1:n-1+max_lag] / (var * n)
    return acf

max_lag = 2000

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for d_idx, (ax, par_name) in enumerate(zip(axes, par_names)):
    for rho in rho_vals:
        pos = traces[rho]['positions'][n_burnin:, d_idx]
        acf = autocorrelation(pos, max_lag=max_lag)
        ax.plot(acf, label=f'ρ={rho}', linewidth=1.5)
    
    ax.axhline(0, color='black', linewidth=0.5)
    ax.set_xlabel('Lag')
    ax.set_ylabel('Autocorrelation')
    ax.set_title(f'ACF: {par_name}')
    ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
# Also show ACF for log-density (measures f-mixing)
fig, ax = plt.subplots(figsize=(10, 5))

for rho in rho_vals:
    ld = traces[rho]['logdensities'][n_burnin:]
    acf = autocorrelation(ld, max_lag=max_lag)
    ax.plot(acf, label=f'ρ={rho}', linewidth=1.5)

ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')
ax.set_title('ACF: log-density (measures f-chain mixing)')
ax.legend()
fig.tight_layout()
plt.show()

## 8. 2D scatter: RKPCN samples vs EP baseline

Compare the RKPCN samples (post-burnin, thinned) against the grid-based EP samples.

In [ ]:
# Grid-based EP samples for reference
ep_samp = saved_samp.get('ep', None)
exact_samp = saved_samp.get('exact', None)

thin = 5

fig, axes = plt.subplots(1, len(rho_vals), figsize=(5*len(rho_vals), 5))

for ax, rho in zip(axes, rho_vals):
    pos = traces[rho]['positions'][n_burnin::thin]
    
    if ep_samp is not None:
        ax.scatter(ep_samp[:, 0], ep_samp[:, 1], alpha=0.15, s=5, c='blue', label='EP (grid)')
    ax.scatter(pos[:, 0], pos[:, 1], alpha=0.3, s=5, c='red', label=f'rkpcn')
    
    ax.set_xlabel(par_names[0])
    if rho == rho_vals[0]:
        ax.set_ylabel(par_names[1])
    ax.set_title(f'ρ={rho}')
    ax.legend(fontsize=9, markerscale=3)

fig.suptitle('RKPCN samples vs grid-based EP', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 9. Effect of more iterations

Re-run ρ=0.99 with 5× more iterations to test whether the issue is simply insufficient chain length.

In [ ]:
# Extended run for rho=0.99
n_total_extended = 275_000  # 5x the standard (250k burnin + 25k post-burnin)
n_burnin_extended = 250_000

jax.clear_caches()
key_extended = jr.key(9999)
trace_extended = run_rkpcn_full_trace(key_extended, rep, rho=0.99, n_total=n_total_extended)

# Compare ESS
pos_std = traces[0.99]['positions'][n_burnin:]
pos_ext = trace_extended['positions'][n_burnin_extended:]

ess_std = compute_ess(pos_std)
ess_ext = compute_ess(pos_ext)

print(f'Standard (5k post-burnin):  ESS = {ess_std}, n_samples = {pos_std.shape[0]}')
print(f'Extended (25k post-burnin): ESS = {ess_ext}, n_samples = {pos_ext.shape[0]}')

In [ ]:
# Compare extended rkpcn99 against EP
thin_ext = 25  # thin more aggressively to get ~1000 samples
pos_ext_thinned = pos_ext[::thin_ext]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Standard rkpcn99
ax = axes[0]
if ep_samp is not None:
    ax.scatter(ep_samp[:, 0], ep_samp[:, 1], alpha=0.15, s=5, c='blue', label='EP')
ax.scatter(pos_std[::thin, 0], pos_std[::thin, 1], alpha=0.3, s=5, c='red', label='rkpcn99')
ax.set_title(f'Standard (n={pos_std[::thin].shape[0]})')
ax.set_xlabel(par_names[0])
ax.set_ylabel(par_names[1])
ax.legend(fontsize=9, markerscale=3)

# Extended rkpcn99
ax = axes[1]
if ep_samp is not None:
    ax.scatter(ep_samp[:, 0], ep_samp[:, 1], alpha=0.15, s=5, c='blue', label='EP')
ax.scatter(pos_ext_thinned[:, 0], pos_ext_thinned[:, 1], alpha=0.3, s=5, c='red', label='rkpcn99 (ext)')
ax.set_title(f'Extended (n={pos_ext_thinned.shape[0]})')
ax.set_xlabel(par_names[0])
ax.legend(fontsize=9, markerscale=3)

# Log-density trace comparison
ax = axes[2]
ld_ext = trace_extended['logdensities']
ax.plot(ld_ext, linewidth=0.2, alpha=0.5)
ax.axvline(n_burnin_extended, color='red', linestyle='--', linewidth=1, label='burnin end')
ax.set_title('Extended log-density trace (ρ=0.99)')
ax.set_xlabel('Iteration')
ax.legend()

fig.tight_layout()
plt.show()

## 10. W2 comparison

Compute W2 distances to grid-based EP for each ρ, including the extended ρ=0.99 run.

In [ ]:
from uncprop.utils.wasserstein import wasserstein2_sinkhorn

if ep_samp is not None:
    print(f'{"method":>20s} | {"W2":>8s} | {"n_samp":>8s} | {"min ESS":>8s}')
    print('-' * 55)
    
    for rho in rho_vals:
        pos = traces[rho]['positions'][n_burnin::thin]
        ess = compute_ess(pos)
        w2 = wasserstein2_sinkhorn(jnp.array(ep_samp), jnp.array(pos),
                                   threshold=1e-6, max_iterations=5000)
        label = 'cut' if rho == 0.0 else f'rkpcn{int(rho*100)}'
        print(f'{label:>20s} | {float(w2):8.4f} | {pos.shape[0]:8d} | {min(ess):8.1f}')
    
    # Extended rkpcn99
    ess_ext = compute_ess(pos_ext_thinned)
    w2_ext = wasserstein2_sinkhorn(jnp.array(ep_samp), jnp.array(pos_ext_thinned),
                                   threshold=1e-6, max_iterations=5000)
    print(f'{"rkpcn99 (extended)":>20s} | {float(w2_ext):8.4f} | {pos_ext_thinned.shape[0]:8d} | {min(ess_ext):8.1f}')
    
    # EUP for reference
    eup_samp = saved_samp.get('eup', None)
    if eup_samp is not None:
        w2_eup = wasserstein2_sinkhorn(jnp.array(ep_samp), jnp.array(eup_samp),
                                       threshold=1e-6, max_iterations=5000)
        print(f'{"eup":>20s} | {float(w2_eup):8.4f} | {eup_samp.shape[0]:8d} | {"N/A":>8s}')
else:
    print('No EP samples available for W2 comparison.')

## 11. Summary

Key takeaways from this replicate:

1. **Log-density traces**: Does ρ=0.99 show the chain stuck at a fixed level (slow f-mixing) while ρ=0 shows rapid fluctuations (fast f-mixing)?

2. **ESS**: How does ESS scale with ρ? Expect ESS to decrease with ρ.

3. **ACF of log-density**: The decorrelation time in log-density directly measures f-mixing speed. Expect ~1/(1-ρ²) scaling.

4. **Extended run**: Does 5× more iterations for ρ=0.99 close the gap with lower-ρ runs? If yes, the fix is simply scaling iterations. If not, there may be deeper issues.

5. **2D scatter**: Does ρ=0.99 cover the same region as EP, just with fewer effective samples? Or is it concentrated in a subregion?